In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns
sns.set_theme(palette="colorblind")

# Ex. 8.1

The code below was used in the examples for the lecture "Forecasting with historical covariates". Modify it to use a DecisionTree Regressor, plot the test-set predictions and output the most important predictors. What are the most significant drivers of real GDP within this model?

Dataset description can be found [here](https://www.statsmodels.org/dev/datasets/generated/macrodata.html).

In [ ]:
from sktime.datasets.forecasting import Macroeconomic

df = Macroeconomic().load()[1]
df.index = df.index.astype('datetime64[ns]') 
df.info()

In [ ]:
df.head()

## Train-test split

In [ ]:
# 80% for training
train_size = int(df.shape[0]*0.8)

train_set = df.iloc[:train_size].copy()
test_set = df.iloc[train_size:].copy()

## Data preprocessing

In [ ]:
# create a seasonal dummy
df["quarter"] = df.index.quarter
train_set["quarter"] = train_set.index.quarter
test_set["quarter"] = test_set.index.quarter

In [ ]:
# diff all variables

train_set = train_set.diff().bfill()
test_set = test_set.diff().bfill()
df = df.diff().bfill()

## Set up TimeSeriesFold

In [ ]:
from skforecast.direct import ForecasterDirectMultiVariate
from skforecast.model_selection import TimeSeriesFold
from skforecast.model_selection import backtesting_forecaster_multiseries
from skforecast.model_selection import grid_search_forecaster_multiseries

???
from sklearn.metrics import root_mean_squared_error

cv = TimeSeriesFold(
         steps=1,
         initial_train_size=train_size,
         refit=False,
         allow_incomplete_fold=True
     )

## Model development

In [ ]:
forecaster = ForecasterDirectMultiVariate(
                 estimator=???,
                 level='realgdp',
                 steps=1,
                 lags=4,
             )

lags_grid = {
    '2 lags': 2,
    '4 lags': 4
}

param_grid = {
    ???
}

cv_hp_search = TimeSeriesFold(
         steps=1,
         initial_train_size=int(train_size*0.8),
         fold_stride=None,
         refit=False
     )

results_search = grid_search_forecaster_multiseries(
    forecaster=forecaster,
    series=train_set.drop(columns=['quarter']),
    exog=train_set['quarter'],
    lags_grid=lags_grid,
    param_grid=param_grid,
    cv=cv_hp_search,
    levels=None,
    metric=[root_mean_squared_error, 'mean_absolute_error'],
    aggregate_metric='weighted_average',
    suppress_warnings=True,
    return_best=True
)

results_search.head(3)

## Model evaluation

In [ ]:
results, predictions = backtesting_forecaster_multiseries(
                                     forecaster=forecaster,
                                     series=df.drop("quarter", axis=1),
                                     exog=df["quarter"],
                                     cv=cv,
                                     metric=[root_mean_squared_error, 'mean_absolute_error']
                                 )
results.index = ["Random Forest"]
results

## Plot predictions

In [ ]:
y_hat = predictions['pred']
tdf = pd.DataFrame({"y": test_set['realgdp'], "y_hat": y_hat})
tdf.plot(figsize=(12,4))

## Feature importances

In [ ]:
forecaster.fit(series=df.drop(columns='quarter'), exog=df['quarter'])
importances = forecaster.get_feature_importances(step=1)
importances

# Ex 8.2

The dataset contains quarterly numbers of overnight trips in different states of Australia, between 1998 and 2018. The trips are categorised according to the purpose of the trip. Complete the following steps:

1. Select data corresponding to business trips 
2. Pivot the table to represent the data in a format that can be used for indepedent multiseries forecasting in skforecast, i.e. date time should be the index, each state should be in a separate column.
3. Create an extra variable to capture seasonality in the data. 
4. Create an independent multi-series model that forecasts quarterly numbers of business trip in each Australian state (use one step ahead forecasts; any ML algorithm of your choice, use no more than 6 HP combinations and no more than 2 lag settings during the grid search).
5. Find out which are the most important predictors of business trips. Provide your comments on the predictors.

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/skforecast/skforecast-datasets/main/data/australia_tourism.csv",
                 parse_dates=["date_time"], index_col="date_time")
df.info()

In [ ]:
df = df[df["Purpose"] == "???"]

# Pivot so each state is a column, each date is a row
pivot = df.pivot_table(index="???", aggfunc="sum",
                       columns='???', values='???',
                       observed=False)

pivot.index.freq = "QS-OCT"

pivot.head(3)

In [ ]:
pivot.shape

In [ ]:
# Create an extra variable to capture seasonality in the data
pivot.loc[:, 'quarter'] = pivot.index.???

## Train-test split

In [ ]:
train_size = int(pivot.shape[0]*0.8)
train_size_cv = int(train_size*0.8)

In [ ]:
train_set = pivot.iloc[:train_size].copy()
test_set = pivot.iloc[train_size:].copy()

## Set up TimeSeriesFold

In [ ]:
from skforecast.model_selection import TimeSeriesFold
from sklearn.metrics import root_mean_squared_error

cv = TimeSeriesFold(
         steps=???,
         initial_train_size=train_size,
         fold_stride=None,
         refit=False
     )

## Hyperparameter optimisation

In [ ]:
from skforecast.recursive import ForecasterRecursiveMultiSeries    
from sklearn.??? import ???


forecaster = ForecasterRecursiveMultiSeries(
    estimator=???,
    lags=4,
    encoding='onehot', # how entity ids are encoded; None - do not use any encoding
)

In [ ]:
X_train, y_train = forecaster.create_train_X_y(
                       series=train_set.drop(columns="quarter"),
                       exog=train_set['quarter']
                   )
X_train.head(3)

In [ ]:
import warnings

from skforecast.model_selection import backtesting_forecaster_multiseries
from skforecast.model_selection import grid_search_forecaster_multiseries
from skforecast.exceptions import InputTypeWarning
warnings.simplefilter('ignore', category=InputTypeWarning)


lags_grid = ???

param_grid = ???

cv_hp_search = TimeSeriesFold(
         steps=???,
         initial_train_size=train_size_cv,
         fold_stride=None,
         refit=False
     )

results_search = grid_search_forecaster_multiseries(
    forecaster=forecaster,
    series=train_set.drop(columns='quarter'),
    exog=train_set['quarter'],
    lags_grid=lags_grid,
    param_grid=param_grid,
    cv=cv_hp_search,
    levels=None,
    metric=[root_mean_squared_error, 'mean_absolute_error'],
    aggregate_metric='weighted_average',
    suppress_warnings=True,
    return_best=True
)

results_search

In [ ]:
best_params = results_search['params'].iat[0]
best_params

In [ ]:
best_lags = results_search['lags'].iat[0]
best_lags

## Model evaluation

In [ ]:
metrics, predictions = backtesting_forecaster_multiseries(
    forecaster=forecaster,
    series=pivot.drop(columns='quarter'),
    exog=pivot['quarter'],
    cv=cv,
    levels=None, # list of levels to calculate metrics; None - all levels
    metric=[root_mean_squared_error, 'mean_absolute_error'],
    add_aggregated_metric=True
)


In [ ]:
metrics

## Important predictors

In [ ]:
forecaster.fit(series=pivot.drop(columns='quarter'), exog=pivot['quarter'])
???

# Ex 8.3

Foundation models are known to recognise seasonal patterns out-of-the-box, i.e. without relying on preprocessing such as seasonal adjustment or one-hot features representing seasonality. Modify the model used in the lecture notebook to include additional features "holiday", "workingday", "weekday, "month" and "hour" that are available in the bike sharing dataset. Examine feature importances and confirm if seasonal dummies do not improve model accuracy.

Install chronos:

pip install chronos-forecasting


In [ ]:
import pandas as pd
import seaborn as sns
sns.set_theme(palette="colorblind")

from skforecast.datasets import fetch_dataset
from skforecast.foundation import FoundationModel, ForecasterFoundation
from skforecast.model_selection import TimeSeriesFold, backtesting_foundation
from sklearn.metrics import root_mean_squared_error

## Load data

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/skforecast/skforecast-datasets/main/data/bike_sharing_dataset_clean.csv", 
                 parse_dates=["date_time"], index_col="date_time")

df = df.asfreq('h')

df.head()

## Train-test split

In [ ]:
# 80% for training
train_size = int(df.shape[0]*0.8)

## Set up TimeSeriesFold

In [ ]:
cv = TimeSeriesFold(
         steps=24,
         initial_train_size=train_size,
         refit=False
     )

## Baseline

In [ ]:
from skforecast.recursive import ForecasterEquivalentDate
from skforecast.model_selection._validation import backtesting_forecaster


last_value_forecaster = ForecasterEquivalentDate(offset=1, n_offsets=1)

results_lv, predictions_backtest = backtesting_forecaster(
                                   forecaster=last_value_forecaster,
                                   y=df['users'],
                                   cv=cv,
                                   metric=[root_mean_squared_error, 
                                           'mean_absolute_error'],
                                   n_jobs='auto',
                                   verbose=False,
                                   show_progress=True
                               )
results_lv.index = ["Last-Value"]
results_lv

## Model development and evaluation

In [ ]:
estimator = FoundationModel(model_id="autogluon/chronos-2-small", context_length=500)
forecaster = ForecasterFoundation(estimator=estimator)

In [ ]:
metrics_chronos, backtest_predictions = backtesting_foundation(
    forecaster=forecaster,
    series=df[???],
    exog=df[???],
    cv=cv,
    metric=[root_mean_squared_error, 'mean_absolute_error'],
    suppress_warnings=True
)

metrics_chronos

In [ ]:
backtest_predictions.head()

In [ ]:
tdf = pd.DataFrame({"y": df.iloc[train_size:, df.columns.get_loc("users")], 
                    "y_hat": backtest_predictions["pred"]})
tdf.plot(figsize=(12,4))

## Feature importances

In [ ]:
# feature ablation

covariates = ???

rmse_main = metrics_chronos.loc[0, "root_mean_squared_error"]
mae_main = metrics_chronos.loc[0, "mean_absolute_error"]

importances_rmse = {}
importances_mae = {}

for feature in covariates:
    X_ablate = df[covariates].copy()
    X_ablate = X_ablate.drop(columns=[feature])

    # make predictions with the ablated feature
    metrics, backtest_predictions = backtesting_foundation(
        forecaster=forecaster,
        series=df['users'],
        exog=X_ablate,
        cv=cv,
        metric=[root_mean_squared_error, 'mean_absolute_error'],
        suppress_warnings=True
    )

    rmse = metrics.loc[0, "root_mean_squared_error"]
    mae = metrics.loc[0, "mean_absolute_error"]

    importances_rmse[feature] = rmse - rmse_main
    importances_mae[feature] = mae - mae_main   

In [ ]:
importances_df = pd.DataFrame({"feature": covariates,
                               "importance_rmse": [importances_rmse[x] for x in covariates],
                               "importance_mae": [importances_mae[x] for x in covariates]})
importances_df.sort_values("importance_rmse", ascending=False)

# Ex. 8.4

Use a multivariate foundation model to predict the number of business trips in each Australian state using the dataset from ex.8.2. Use the context window of 40 observations. Plot the predictions and describe your observations. How does the accuracy of the foundation model compare to the accuracy of ML models from ex. 8.2?

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/skforecast/skforecast-datasets/main/data/australia_tourism.csv",
                 parse_dates=["date_time"], index_col="date_time")
df.info()

In [ ]:
df = df[df["Purpose"] == "???"]

# Pivot so each state is a column, each date is a row
pivot = df.pivot_table(index="???", aggfunc="sum",
                       columns='???', values='Trips',
                       observed=False)

pivot.index.freq = "QS-OCT"

pivot.head(3)

In [ ]:
# train set size
train_size = int(pivot.shape[0]*0.8)

In [ ]:
from skforecast.model_selection import TimeSeriesFold
from sklearn.metrics import root_mean_squared_error

cv = TimeSeriesFold(
         steps=???,
         initial_train_size=train_size,
         fold_stride=None,
         refit=False
     )

In [ ]:
from skforecast.foundation import FoundationModel, ForecasterFoundation
from skforecast.model_selection import backtesting_foundation

In [ ]:
estimator = FoundationModel(model_id="autogluon/chronos-2-small", context_length=???)
forecaster = ForecasterFoundation(estimator=estimator)

In [ ]:
metrics_chronos, backtest_predictions = backtesting_foundation(
    forecaster=forecaster,
    series=???,
    levels=pivot.columns.tolist(),
    cv=cv,
    metric=[root_mean_squared_error, 'mean_absolute_error'],
    suppress_warnings=True
)

metrics_chronos

In [ ]:
# for each item
for item in pivot.columns:
    y_hat = backtest_predictions[backtest_predictions["level"] == item]['pred']
    tdf = pd.DataFrame({"y": pivot.loc[y_hat.index, item], "y_hat": y_hat})
    tdf.plot(figsize=(12,2), title=item)

# Citing this notebook

If you use this notebook in your work, please cite it as follows:
    
Pekar, V. (2026). Business Forecasting. Lecture examples and exercises. (Version 1.0.0). URL: https://github.com/vpekar/bf